In [4]:
import os
import csv
import datetime as dt
import pandas as pd

# index를 날짜로 하는 데이터프레임을 price, divend에 대해 반환
def extract_stock_data(file_path, ticker):
    extract_datas = []

    with open(os.path.join(file_path, ticker + '.csv'), newline='') as csvfile:
        reader = csv.reader(csvfile)

        for row in reader:
            extract_datas.append(row)
        
    price_dates = list(map(dt.date.fromisoformat, extract_datas[0]))
    price_history = list(map(float, extract_datas[1]))
    divend_dates = list(map(dt.date.fromisoformat, extract_datas[2]))
    divend_history = list(map(float, extract_datas[3]))

    price_df = pd.DataFrame({'date': price_dates, 'price': price_history})
    divend_df = pd.DataFrame({'date': divend_dates, 'divend': divend_history})

    price_df.set_index('date', inplace=True)
    divend_df.set_index('date', inplace=True)

    return (price_df, divend_df)


In [5]:
def make_stock_data(file_path, tickers):
    prices = []
    divends = []

    for ticker in tickers:
        price_df, divend_df = extract_stock_data(file_path, ticker)
        price_df.rename(columns={'price': ticker}, inplace=True)
        divend_df.rename(columns={'divend': ticker}, inplace=True)
        prices.append(price_df)
        divends.append(divend_df)

    stock_price_df = pd.concat(prices, axis=1, sort=True)
    stock_divend_df = pd.concat(divends, axis=1, sort=True)

    return {'price': stock_price_df, 'divend': stock_divend_df}

In [ ]:
import itertools

class Portfolio:
    # stock_data
    def __init__(self, stock_data: dict, start_cash: float, target_ratio: dict, buy_ratio: float = 5.0, sell_ratio: float = 5.0):
        self.__stock_list = stock_data['price'].columns.to_list()
        self.__price_data = stock_data['price'].copy()
        self.__divend_data = stock_data['divend'].copy()
        self.__start_cash = start_cash
        # target_ratio의 구성 { stock명 : 목표 비중, ... }
        self.__target_ratio = target_ratio
        self.__buy_ratio = buy_ratio
        self.__sell_ratio = sell_ratio
        self.__target_buy_ratio = {ticker: min(target_ratio[ticker] * buy_ratio / 100, buy_ratio / 100) for ticker in self.stock_list}
        self.__target_sell_ratio = {ticker: min(target_ratio[ticker] * sell_ratio / 100, sell_ratio / 100) for ticker in self.stock_list}
    

        self.__price_data.dropna(inplace=True)
        self.__avilable_date = self.price_data.iloc[0].name
        self.start_date = self.__avilable_date
        self.__divend_data = self.__divend_data[self.__divend_data.index >= self.start_date]
        
    @property
    def stock_list(self) -> list:
        return self.__stock_list
    @property
    def price_data(self) -> pd.DataFrame:
        return self.__price_data
    @property
    def divend_data(self) -> pd.DataFrame:
        return self.__divend_data
    @property
    def target_ratio(self) -> dict:
        return self.__target_ratio
    @target_ratio.setter
    def target_ratio(self, value):
        self.__target_ratio = value
        self.__target_buy_ratio = {ticker: min(value[ticker] * self.__buy_ratio / 100, self.__buy_ratio / 100) for ticker in self.stock_list}
        self.__target_sell_ratio = {ticker: min(value[ticker] * self.__sell_ratio / 100, self.__sell_ratio / 100) for ticker in self.stock_list}
    @property
    def buy_ratio(self) -> float:
        return self.__buy_ratio
    @buy_ratio.setter
    def buy_ratio(self, value):
        self.__buy_ratio = value
        self.__target_buy_ratio = {ticker: min(self.__target_ratio[ticker] * value / 100, value / 100) for ticker in self.stock_list}
    @property
    def sell_ratio(self) -> float:  
        return self.__sell_ratio
    @sell_ratio.setter
    def sell_ratio(self, value):
        self.__sell_ratio = value
        self.__target_sell_ratio = {ticker: min(self.__target_ratio[ticker] * value / 100, value / 100) for ticker in self.stock_list}

    def backtest(self):
        info_list =  ['number', 'value', 'weight']
        stock_info =  list(itertools.product(self.stock_list, info_list))

        col = [('total', 'value'), ('cash', 'value')] + stock_info
        col = pd.MultiIndex.from_tuples(col)
        dates = self.price_data.index
        stat = pd.DataFrame(columns=col, index=dates)

        # 첫 값 설정
        first_date = dates[0]
        total_value = self.__start_cash
        cash_value = 0
        for ticker in self.stock_list:
            stat.loc[first_date, ('total', 'value')] = total_value
            stat.loc[first_date, ('cash', 'value')] = cash_value
            stat.loc[first_date, [ticker]] = self.__ideal_nvw(total_value, ticker, first_date)
            
        for i in range(1, len(dates)):
            # 가격 변동 처리
            prev_num = stat.loc[dates[i - 1]][:, 'number']
            prev_num_pv = (prev_num * self.price_data.loc[dates[i]]).round(3)
            total_value = prev_num_pv.sum()
            # 배당금 처리
            divend = self.acculate_divend(dates[i-1], dates[i])
            total_value += divend

            break
        #     # 변화된 금액에 따른 가격 처리
        #     for ticker in self.stock_list:
        #         continue
        # print(stat)

    # 범위는 (start_date, end_date]
    def acculate_divend(self, start_date, end_date):
        divend_df = self.divend_data[(self.divend_data.index > start_date) & (self.divend_data.index <= end_date)]
        return divend_df.sum().sum()
    
    def __ideal_nvw(self, total_value, ticker, date):
        price = self.price_data.loc[date][ticker]
        target_ratio = self.target_ratio[ticker]
        value = round(total_value * target_ratio, 3)
        number = round(value / price, 6)
        return (number, value, target_ratio)
        self.__
    # def __real_nvw(self, total_value, ticker, date):
    #     price = self.price_data.loc[date][ticker]
    #     ideal = self.__ideal_nvw(total_value, ticker, date)
        


In [245]:
# stock 불러와서 데이터프레임화 하기
file_path = './etf'
tickers = ['QQQ', 'DGRW', 'SCHD']

stock_data = make_stock_data(file_path, tickers)

In [246]:
portfolio = Portfolio(stock_data, 1000000, {'QQQ': 0.5, 'DGRW': 0.3, 'SCHD': 0.2})

portfolio.backtest()